<a href="https://colab.research.google.com/github/luvr1/CCMACLRL_EXERCISES/blob/main/Exercise6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Exercise 6: Choosing the best performing model on a dataset

Instructions:

- Use the Dataset File to train your model
- Use the Test File to generate your results
- Use the Sample Submission file to generate the same format
- Use all Regression models

Submit your results to:
https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview



In [2]:
import pandas as pd
import seaborn as sns

from matplotlib import pyplot as plt
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

import numpy as np
from sklearn.model_selection import cross_val_score

## Dataset File

In [3]:
train_data = 'https://github.com/robitussin/CCMACLRL_EXERCISES/blob/3fd7d51ffd17863598ac3f44eeefc558171a5b73/dataset/house-prices-advanced-regression-techniques/train.csv?raw=true'
df = pd.read_csv(train_data)

In [4]:
X = df.drop(['SalePrice', 'Id'], axis=1)
y = df['SalePrice']

## Test File

In [5]:
test_url = 'https://github.com/robitussin/CCMACLRL_EXERCISES/blob/3fd7d51ffd17863598ac3f44eeefc558171a5b73/dataset/house-prices-advanced-regression-techniques/test.csv?raw=true'
dt=pd.read_csv(test_url)

In [6]:
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1459 non-null   int64  
 1   MSSubClass     1459 non-null   int64  
 2   MSZoning       1455 non-null   object 
 3   LotFrontage    1232 non-null   float64
 4   LotArea        1459 non-null   int64  
 5   Street         1459 non-null   object 
 6   Alley          107 non-null    object 
 7   LotShape       1459 non-null   object 
 8   LandContour    1459 non-null   object 
 9   Utilities      1457 non-null   object 
 10  LotConfig      1459 non-null   object 
 11  LandSlope      1459 non-null   object 
 12  Neighborhood   1459 non-null   object 
 13  Condition1     1459 non-null   object 
 14  Condition2     1459 non-null   object 
 15  BldgType       1459 non-null   object 
 16  HouseStyle     1459 non-null   object 
 17  OverallQual    1459 non-null   int64  
 18  OverallC

## Sample Submission File

In [11]:
sample_submission_url ='https://github.com/robitussin/CCMACLRL_EXERCISES/blob/3fd7d51ffd17863598ac3f44eeefc558171a5b73/dataset/house-prices-advanced-regression-techniques/sample_submission.csv?raw=true'

sf=pd.read_csv(sample_submission_url)

In [12]:
sf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Id         1459 non-null   int64  
 1   SalePrice  1459 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 22.9 KB


In [32]:
X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

num_features = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_features = X_train.select_dtypes(include=["object"]).columns

num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_transformer, num_features),
    ("cat", cat_transformer, cat_features)
])

In [31]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## 1. Train a KNN Regressor

In [35]:
knn = KNeighborsRegressor(n_neighbors=5)

knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", knn)
])

knn_pipeline.fit(X_train, y_train)
train_score = knn_pipeline.score(X_val, y_val)
print("KNN Score is:", train_score)


KNN Score is: 0.8034594460370994


- Perform cross validation

In [36]:
cv_scores = cross_val_score(knn_pipeline, X_train, y_train, cv=5, scoring="neg_mean_squared_error")
rmse_scores = np.sqrt(-cv_scores)
print("Cross-validation KNN score is:", rmse_scores.mean())

Cross-validation KNN score is: 37591.31956226484


## 2. Train a SVM Regression

In [37]:
svm = SVR(kernel="rbf", C=100, gamma=0.1, epsilon=0.2)

svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", svm)
])

svm_pipeline.fit(X_train, y_train)
train_score = svm_pipeline.score(X_val, y_val)
print("SVM Score is:", train_score)

SVM Score is: -0.021736807908757205


- Perform cross validation

In [41]:
cv_scores = cross_val_score(svm_pipeline, X_train, y_train, cv=5, scoring="neg_mean_squared_error")
rmse_scores = np.sqrt(-cv_scores)
print("SVM cross-validation core is:", rmse_scores.mean())


SVM cross-validation core is: 78731.27823351705


## 3. Train a Decision Tree Regression

In [39]:
dt_model = DecisionTreeRegressor(random_state=42)

dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", dt_model)
])

dt_pipeline.fit(X_train, y_train)
dt_score = dt_pipeline.score(X_val, y_val)
print("Decision Tree Score is:", dt_score)

Decision Tree Score is: 0.7716297030408461


- Perform cross validation

In [40]:
dt_cv_scores = cross_val_score(dt_pipeline, X_train, y_train, cv=5, scoring="neg_mean_squared_error")
dt_rmse = np.sqrt(-dt_cv_scores).mean()
print("Cross Validation Decision Tree score is:", dt_rmse)

Cross Validation Decision Tree score is: 47285.423686371745


## 4. Train a Random Forest Regression

In [42]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", rf_model)
])

rf_pipeline.fit(X_train, y_train)
rf_score = rf_pipeline.score(X_val, y_val)
print("Random Forest score is:", rf_score)


Random Forest score is: 0.8939002860298636


In [43]:
rf_cv_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring="neg_mean_squared_error")
rf_rmse = np.sqrt(-rf_cv_scores).mean()
print("Random Forest score-validation is:", rf_rmse)

Random Forest score-validation is: 30694.620730383263


## 5. Compare all the performance of all regression models

In [45]:
results = {
    "Model": ["KNN Regressor", "SVM Regressor", "Decision Tree", "Random Forest"],
    "Validation R²": [
        knn_pipeline.score(X_val, y_val),
        svm_pipeline.score(X_val, y_val),
        dt_pipeline.score(X_val, y_val),
        rf_pipeline.score(X_val, y_val)
    ]
}

print("\nModel Performance Comparison (Validation R²):")
print(results_df)


Model Performance Comparison (Validation R²):
           Model  Validation R²
0  KNN Regressor       0.803459
1  SVM Regressor      -0.021737
2  Decision Tree       0.771630
3  Random Forest       0.893900


## 6. Generate Submission File

Choose the model that has the best performance to generate a submission file.

In [29]:

X_test_processed = preprocessor.transform(dt)

y_pred = rf_pipeline.predict(dt)

submission_df = pd.DataFrame({
    'Id': dt['Id'],
    'SalePrice': y_pred
})
submission_df.to_csv('submission_file.csv', index=False)
print("Submission file created: submission_file.csv")

from google.colab import files
files.download('submission_file.csv')

Submission file created: submission_file.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>